# **Baseline 35 feature models**

In [1]:
!pip install catboost
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, accuracy_score, recall_score,
                             f1_score, precision_score, roc_auc_score, confusion_matrix,
                             roc_curve, auc, precision_recall_curve, ConfusionMatrixDisplay,
                             average_precision_score)
from sklearn.utils import resample

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00


In [2]:
data = pd.read_excel('/content/dataset.xlsx')
print("Shape: ", data.shape)

features_df    = pd.read_csv('/content/FINAL_35_features_selected.csv')
FINAL_FEATURES = features_df['feature'].tolist()
N_FEATURES     = len(FINAL_FEATURES)

X = data[FINAL_FEATURES]
y = data['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("Distributia datelor:")
print(f"X_train shape: {X_train.shape} ({X_train.shape[0]} paciente, {X_train.shape[1]} simptome)")
print(f"X_test shape:  {X_test.shape} ({X_test.shape[0]} paciente, {X_test.shape[1]} simptome)")
print(f"Prevalența bolii (Train): {y_train.mean()*100:.1f}%")
print(f"Prevalența bolii (Test):  {y_test.mean()*100:.1f}%")

Shape:  (886, 61)
Distributia datelor:
X_train shape: (708, 35) (708 paciente, 35 simptome)
X_test shape:  (178, 35) (178 paciente, 35 simptome)
Prevalența bolii (Train): 53.5%
Prevalența bolii (Test):  53.4%


In [3]:
def evaluate_model(y_true, y_pred, y_proba, model_name="Model"):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  #Recall
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0

    acc   = accuracy_score(y_true, y_pred)
    f1    = f1_score(y_true, y_pred, zero_division=0)
    auc_roc = roc_auc_score(y_true, y_proba)
    auprc   = average_precision_score(y_true, y_proba)

    print(f"========== PERFORMANȚĂ: {model_name} ==========")
    print(f"AUC-ROC:     {auc_roc:.4f}")
    print(f"AUPRC:       {auprc:.4f}")
    print(f"Accuracy:    {acc:.4f}")
    print(f"F1-Score:    {f1:.4f}")
    print(f"Sensitivity: {sensitivity:.4f} (Recall / True Positive Rate)")
    print(f"Specificity: {specificity:.4f} (True Negative Rate)")
    print(f"Precision:   {precision:.4f} (Positive Predictive Value)")
    print("-------------------------------------------------")
    print("Confusion Matrix:")
    print(f"[{tn}] TN   [{fp}] FP")
    print(f"[{fn}] FN   [{tp}] TP\n")

    return {'auc': auc_roc, 'auprc': auprc, 'acc': acc, 'f1': f1,
            'sens': sensitivity, 'spec': specificity, 'prec': precision}

**SVM RBF**

In [5]:
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("ANTRENARE MODEL: SVM RBF")
best_svm_rbf = SVC(
    kernel='rbf',
    class_weight='balanced',
    probability=True,
    random_state=42
)
best_svm_rbf.fit(X_train, y_train)

cv_outer_rbf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_rbf = cross_val_score(
    best_svm_rbf,
    X_train,
    y_train,
    cv=cv_outer_rbf,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_rbf.mean():.4f} ± {cv_scores_rbf.std():.4f}\n")

y_pred_svm_rbf  = best_svm_rbf.predict(X_test)
y_proba_svm_rbf = best_svm_rbf.predict_proba(X_test)[:, 1]

svm_rbf_test_metrics = evaluate_model(
    y_test,
    y_pred_svm_rbf,
    y_proba_svm_rbf,
    model_name="SVM (RBF)"
)

ANTRENARE MODEL: SVM RBF
10-Fold CV (Train) AUC-ROC: 0.9756 ± 0.0170

========== PERFORMANȚĂ: SVM (RBF) ==========
AUC-ROC:     0.9760
AUPRC:       0.9807
Accuracy:    0.9157
F1-Score:    0.9198
Sensitivity: 0.9053 (Recall / True Positive Rate)
Specificity: 0.9277 (True Negative Rate)
Precision:   0.9348 (Positive Predictive Value)
-------------------------------------------------
Confusion Matrix:
[77] TN   [6] FP
[9] FN   [86] TP



**SVM LINEAR**

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("ANTRENARE MODEL: SVM RBF")
best_svm_linear = SVC(
    kernel='linear',
    class_weight='balanced',
    probability=True,
    random_state=42
)
best_svm_linear.fit(X_train, y_train)

cv_outer_linear = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_linear = cross_val_score(
    best_svm_linear,
    X_train,
    y_train,
    cv=cv_outer_linear,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_linear.mean():.4f} ± {cv_scores_linear.std():.4f}\n")

y_pred_svm_linear  = best_svm_linear.predict(X_test)
y_proba_svm_linear = best_svm_linear.predict_proba(X_test)[:, 1]
svm_linear_test_metrics = evaluate_model(
    y_test,
    y_pred_svm_linear,
    y_proba_svm_linear,
    model_name="SVM (Linear)"
)

ANTRENARE MODEL: SVM RBF
10-Fold CV (Train) AUC-ROC: 0.9674 ± 0.0237

========== PERFORMANȚĂ: SVM (Linear) ==========
AUC-ROC:     0.9750
AUPRC:       0.9802
Accuracy:    0.8933
F1-Score:    0.8962
Sensitivity: 0.8632 (Recall / True Positive Rate)
Specificity: 0.9277 (True Negative Rate)
Precision:   0.9318 (Positive Predictive Value)
-------------------------------------------------
Confusion Matrix:
[77] TN   [6] FP
[13] FN   [82] TP



**RANDOM FOREST**

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("ANTRENARE MODEL: RANDOM FOREST")

best_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
best_rf.fit(X_train, y_train)
cv_outer_rf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_rf = cross_val_score(
    best_rf,
    X_train,
    y_train,
    cv=cv_outer_rf,
    scoring='roc_auc',
    n_jobs=1
)
print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}\n")
y_pred_rf  = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

#y_proba_rf = best_rf.predict_proba(X_test)[:, 1]
#y_pred_rf  = (y_proba_rf > 0.5).astype(int)

rf_test_metrics = evaluate_model(
    y_test,
    y_pred_rf,
    y_proba_rf,
    model_name="Random Forest"
)

ANTRENARE MODEL: RANDOM FOREST
10-Fold CV (Train) AUC-ROC: 0.9709 ± 0.0151

========== PERFORMANȚĂ: Random Forest ==========
AUC-ROC:     0.9762
AUPRC:       0.9807
Accuracy:    0.9326
F1-Score:    0.9362
Sensitivity: 0.9263 (Recall / True Positive Rate)
Specificity: 0.9398 (True Negative Rate)
Precision:   0.9462 (Positive Predictive Value)
-------------------------------------------------
Confusion Matrix:
[78] TN   [5] FP
[7] FN   [88] TP



**XGBOOST**

In [20]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

print("ANTRENARE MODEL: XGBOOST")

ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

best_xgb = XGBClassifier(
    scale_pos_weight=ratio,
    random_state=42,
    n_jobs=-1
)

best_xgb.fit(X_train, y_train)

cv_outer_xgb = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_xgb = cross_val_score(
    best_xgb,
    X_train,
    y_train,
    cv=cv_outer_xgb,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_xgb.mean():.4f} ± {cv_scores_xgb.std():.4f}\n")

#y_pred_xgb  = best_xgb.predict(X_test)
#y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]
y_pred_xgb  = best_xgb.predict(X_test)
y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]


xgb_test_metrics = evaluate_model(
    y_test,
    y_pred_xgb,
    y_proba_xgb,
    model_name="XGBoost"
)

ANTRENARE MODEL: XGBOOST
10-Fold CV (Train) AUC-ROC: 0.9739 ± 0.0139

========== PERFORMANȚĂ: XGBoost ==========
AUC-ROC:     0.9768
AUPRC:       0.9822
Accuracy:    0.9101
F1-Score:    0.9184
Sensitivity: 0.9474 (Recall / True Positive Rate)
Specificity: 0.8675 (True Negative Rate)
Precision:   0.8911 (Positive Predictive Value)
-------------------------------------------------
Confusion Matrix:
[72] TN   [11] FP
[5] FN   [90] TP



**CATBOOST**

In [21]:
!pip install catboost

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("ANTRENARE MODEL: CATBOOST")

best_catboost = CatBoostClassifier(
    scale_pos_weight=float(ratio),
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

best_catboost.fit(X_train, y_train)
cv_outer_cat = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_cat = cross_val_score(
    best_catboost,
    X_train,
    y_train,
    cv=cv_outer_cat,
    scoring='roc_auc',
    n_jobs=-1
)
print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_cat.mean():.4f} ± {cv_scores_cat.std():.4f}\n")
y_pred_cat  = best_catboost.predict(X_test)
y_proba_cat = best_catboost.predict_proba(X_test)[:, 1]
catboost_test_metrics = evaluate_model(
    y_test,
    y_pred_cat,
    y_proba_cat,
    model_name="CatBoost"
)

ANTRENARE MODEL: CATBOOST
10-Fold CV (Train) AUC-ROC: 0.9742 ± 0.0132

========== PERFORMANȚĂ: CatBoost ==========
AUC-ROC:     0.9763
AUPRC:       0.9809
Accuracy:    0.9101
F1-Score:    0.9149
Sensitivity: 0.9053 (Recall / True Positive Rate)
Specificity: 0.9157 (True Negative Rate)
Precision:   0.9247 (Positive Predictive Value)
-------------------------------------------------
Confusion Matrix:
[76] TN   [7] FP
[9] FN   [86] TP

